# 04. 판정 규칙 검증 (2026-11-17)

판정 없이 사실만 정리한다. 03의 판정 함수(`src/judge.py`)를 사람 판정 표본과 대조한다.

| 표본 | 대조 | 상태 |
|---|---|---|
| 범위밖_분류표 32건 | 사람 판정 규칙(R3-HUMAN)을 뺀 자동 규칙의 처리·사유 코드 vs 사람 판정 | 아래 1 |
| 원천검수표 20건 | body_grade 기준 '원천 확인 필요' vs 사람 판정(오류 4, 오탐 16) | 아래 2 |
| 이관검수표 20건 | 규칙 이관 분류·판정 vs 사람 판정 | 표본 생성(3), 비교 셀(4)은 사람 판정 후 실행 |

In [ ]:
import sys, re
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)
from src.load_cot import load_cot, load_products
from src.judge import judge
AS_OF = "2026-11-18"  # D57 포함
cot = load_cot(); prod = load_products()
j, ctx = judge(cot, prod, as_of=AS_OF)
D57_RULES = ("R3-D57-DMG", "R3-D57-3", "R3-D57-4", "R3-D57-2")
j_auto, _ = judge(cot, prod, as_of=AS_OF, drop_rules=("R3-HUMAN",), ctx=ctx)            # D57 추가 후
j_auto0, _ = judge(cot, prod, as_of=AS_OF, drop_rules=("R3-HUMAN",) + D57_RULES, ctx=ctx)  # D57 추가 전
rd = lambda p: pd.read_csv(ROOT / "data" / "review" / p, encoding="utf-8-sig", dtype=str, keep_default_na=False)
print("판정:", len(j), "/ 자동 규칙만:", len(j_auto))

## 1. 범위밖_분류표 32건: 자동 규칙만 vs 사람 판정

- 사람 처리: '제외' 또는 '엄격본 제외…' → 제외, 그 밖(194 '교정 + 태그') → 제외 아님.
- 사유 코드: 사람 '내 분류'(1~4 또는 글자 값) vs 자동 규칙의 사유 코드(D30·D52·D55). 글자가 같을 때만 일치로 센다.
- 애매한 경우 처리: 자동 규칙에는 코드 1(고객 요청·경험 이유)을 붙이는 규칙이 없다. 그래서 코드 1은 자동으로 일치할 수 없다.

In [ ]:
rv = rd("범위밖_분류표.csv")
rv["cot_id"] = rv["cot_id"].astype(int)
rv["사람 처리"] = rv["처리"].map(lambda v: "제외" if v == "제외" or v.startswith("엄격본 제외") else v)

def compare32(jj):
    a = rv[["cot_id", "사람 처리", "내 분류", "판단"]].merge(
        jj[["cot_id", "분할", "기준③ 판정", "기준③ 처리", "사유 코드", "적용 규칙"]], on="cot_id")
    a["처리 일치"] = (a["사람 처리"] == "제외") == (a["기준③ 처리"] == "제외")
    a["사유 일치"] = a["내 분류"] == a["사유 코드"]
    a["사람 확인"] = a["기준③ 판정"].str.startswith("판정 불가")
    return a

a0 = compare32(j_auto0)
print("[D57 추가 전]", f"처리 일치 {a0['처리 일치'].sum()}/{len(a0)}, 사유 일치 {a0['사유 일치'].sum()}/{len(a0)}, 둘 다 {(a0['처리 일치'] & a0['사유 일치']).sum()}, 판정 불가 {a0['사람 확인'].sum()}")
a = compare32(j_auto)
print("[D57 추가 후]")
print("제외 총수 (사람 판정 포함 전체 규칙):", (j["기준③ 처리"] == "제외").sum(), "/ 자동 규칙만 — D57 전:", (j_auto0["기준③ 처리"] == "제외").sum(), "후:", (j_auto["기준③ 처리"] == "제외").sum())
print("대상:", len(a))
print(f"처리 일치: {a['처리 일치'].sum()} / {len(a)} ({a['처리 일치'].mean() * 100:.1f}%)")
print(f"사유 코드 일치: {a['사유 일치'].sum()} / {len(a)} ({a['사유 일치'].mean() * 100:.1f}%)")
print(f"둘 다 일치: {(a['처리 일치'] & a['사유 일치']).sum()} / {len(a)}")
print()
print(pd.crosstab(a["내 분류"], a["사유 코드"], margins=True))

In [ ]:
# 다른 건 목록 (처리 또는 사유 코드가 다름)
a.loc[~(a["처리 일치"] & a["사유 일치"]),
      ["cot_id", "분할", "사람 처리", "내 분류", "판단", "기준③ 판정", "기준③ 처리", "사유 코드", "적용 규칙"]]

## 2. 원천검수표 20건: body_grade '원천 확인 필요' vs 사람 판정

- 원천검수표는 H1(상품명 키워드와 설명 불일치)로 뽑은 표본이다. body_grade는 '설명 속 운용사 등급 ≠ 등록 등급'을 본다. 보는 대상이 다르다.
- 사람 판정: 오류 4건(다른 상품 설명 2, 등급 과소 기재 1, 형제 상품 설명 재사용 1), 일치(오탐) 16건.

In [ ]:
from src.extract import body_grade, grade_num
sv = rd("원천검수표.csv")
ps = prod[prod["domain"] == "증권"][["product_name", "risk_grade", "investment_strategy"]].copy()
ps["설명 속 등급"] = ps["investment_strategy"].map(body_grade)
ps["등록 등급"] = ps["risk_grade"].map(grade_num)
b = sv[["상품명", "등급", "내 판정"]].merge(ps[["product_name", "등록 등급", "설명 속 등급"]],
                                     left_on="상품명", right_on="product_name", how="left")
print("매칭 안 된 상품:", b["product_name"].isna().sum())
b["원천 확인 필요"] = b["설명 속 등급"].map(lambda x: isinstance(x, int)) & (b["설명 속 등급"].astype(str) != b["등록 등급"].map(lambda g: str(int(g)) if pd.notna(g) else ""))
b["설명 속 등급 상태"] = b["설명 속 등급"].map(lambda x: "있음" if isinstance(x, int) else x)
b["사람"] = b["내 판정"].map(lambda v: "일치(오탐)" if v == "일치(오탐)" else "오류")
print(pd.crosstab(b["사람"], b["원천 확인 필요"], margins=True))
print()
print(pd.crosstab(b["내 판정"], [b["설명 속 등급 상태"], b["원천 확인 필요"]], margins=True))
b[["상품명", "내 판정", "등록 등급", "설명 속 등급", "원천 확인 필요"]].sort_values(["내 판정", "원천 확인 필요"])

## 3. 이관검수표 20건 생성 → `data/review/이관검수표.csv`

- 성향 밖 등급 언급 15건 + 성향 안 언급 5건, 무작위 (random_state=0). 희망검수표 1·2에 쓴 CoT(40건)는 뺀다.
  - 성향 밖: 03 판정의 '성향 밖 언급'이 있는 CoT (해석 등급 또는 R4 문자 등급이 D43 범위 밖).
  - 성향 안: 등급 언급이 있고 D21 성향이 있으며 성향 밖 언급이 없는 CoT.
- 열: cot_id, 분할, 성향, 언급 등급, 언급 문장, 추천 등급, 내 분류, 내 판정. 규칙 결과는 넣지 않는다 (검수에 영향 주지 않도록).
- 이미 있으면 덮어쓰지 않는다. 원문이 있어 git 제외 폴더에 둔다.

In [ ]:
from src.extract import _mask_ranges
from src.transfer import sentence_mentions
qtext = cot.set_index("cot_id")["question"]
used = set()
for f in ["희망검수표.csv", "희망검수표2.csv"]:
    used |= set(rd(f)["cot_id"].astype(int))
base = j[(j["이관 분류"] != "언급 없음") & ~j["cot_id"].isin(used)]
p_out = base[base["성향 밖 언급"].fillna("") != ""]
p_in = base[(base["성향 밖 언급"].fillna("") == "") & (ctx["cots"].loc[base["cot_id"], "성향_D21"].values != "비교 불가")]
print("모집단 — 성향 밖:", len(p_out), "/ 성향 안:", len(p_in), "/ 뺀 CoT:", len(used))
s = pd.concat([p_out.sample(15, random_state=0), p_in.sample(5, random_state=0)])
rec_g = ctx["recs"].groupby("cot_id")["json_grade"].agg(lambda g: "|".join(str(int(x)) if pd.notna(x) else "?" for x in g))
rows = []
for cid in s["cot_id"]:
    sents = [x for x in re.split(r"(?<=[.!?])\s+", _mask_ranges(str(qtext[cid]))) if sentence_mentions(x)]
    grades = sorted({m[0] for x in sents for m in sentence_mentions(x)})
    rows.append({"cot_id": cid, "분할": ctx["cots"].loc[cid, "split"], "성향": ctx["cots"].loc[cid, "성향_D21"],
                 "언급 등급": ",".join(map(str, grades)), "언급 문장": " / ".join(sents),
                 "추천 등급": rec_g.get(cid, ""), "내 분류": "", "내 판정": ""})
review7 = pd.DataFrame(rows).sort_values("cot_id")
p7 = ROOT / "data" / "review" / "이관검수표.csv"
if p7.exists():
    print("이미 있어 덮어쓰지 않음:", p7)
else:
    review7.to_csv(p7, index=False, encoding="utf-8-sig")
    print("저장:", p7, "/ 행 수:", len(review7))
print(review7[["cot_id", "분할", "성향", "언급 등급", "추천 등급"]].to_string(index=False))
print("성향 밖 15 / 성향 안 5:", s["cot_id"].isin(p_out["cot_id"]).sum(), "/", s["cot_id"].isin(p_in["cot_id"]).sum())

## 4. 이관검수표 비교 (사람 판정 후 실행)

- '내 분류'는 희망(명확) / 고려 중 / 경험 / 비교 표현 / 회피 / 오탐(일반 표현) 중 하나, '내 판정'은 결함 / 결함 아님으로 적는다고 가정한다.
  다른 값이면 아래 MAP을 고친다.
- 규칙 쪽: 03 판정의 '이관 분류'(CoT 단위)와 '이관 판정'.

In [ ]:
t7 = rd("이관검수표.csv")
if (t7["내 분류"] == "").all() and (t7["내 판정"] == "").all():
    print("아직 사람 판정 없음:", len(t7), "건")
else:
    t7["cot_id"] = t7["cot_id"].astype(int)
    MAP = {"희망(명확)": "R1 명확한 희망", "고려 중": "R2 고려 중", "경험": "경험", "비교 표현": "R4 비교 표현", "회피": "회피"}
    c7 = t7.merge(j[["cot_id", "이관 분류", "이관 판정", "성향 밖 언급", "사람 확인 사유"]], on="cot_id")
    c7["분류 일치"] = c7["내 분류"].map(MAP) == c7["이관 분류"]
    c7["판정 일치"] = (c7["내 판정"] == "결함") == (c7["이관 판정"] == "결함")
    print("입력된 행:", (c7["내 분류"] != "").sum())
    print(pd.crosstab(c7["내 분류"], c7["이관 분류"], margins=True)); print()
    print(pd.crosstab(c7["내 판정"], c7["이관 판정"], margins=True)); print()
    print(f"분류 일치 {c7['분류 일치'].sum()} / {len(c7)}, 판정 일치 {c7['판정 일치'].sum()} / {len(c7)}")
    print(f"규칙 '사람 확인' {(c7['이관 판정'] == '사람 확인').sum()}건")
    print(c7.loc[~(c7["분류 일치"] & c7["판정 일치"]), ["cot_id", "성향", "언급 등급", "내 분류", "이관 분류", "내 판정", "이관 판정", "사람 확인 사유"]].to_string(index=False))

## 5. 04 결과 요약

- 일치: 범위밖 = 처리와 사유 코드가 모두 같음, 원천 = '원천 확인 필요' 표시가 사람의 오류/오탐과 같음, 이관 = 분류(MAP)와 결함 판정이 모두 같음.
- 사람 확인: 자동 규칙이 판정하지 않고 사람에게 넘긴 수 (범위밖 = '판정 불가', 원천 = 없음, 이관 = '사람 확인').

In [ ]:
t7c = rd("이관검수표.csv"); t7c["cot_id"] = t7c["cot_id"].astype(int)
MAP7 = {"희망(명확)": "R1 명확한 희망", "고려 중": "R2 고려 중", "경험": "경험", "비교 표현": "R4 비교 표현", "회피": "회피"}
c7s = t7c.merge(j[["cot_id", "이관 분류", "이관 판정"]], on="cot_id")
c7s["분류 일치"] = c7s["내 분류"].map(MAP7) == c7s["이관 분류"]
c7s["판정 일치"] = (c7s["내 판정"] == "결함") == (c7s["이관 판정"] == "결함")
rows = [
    ("범위밖 (D57 전)", len(a0), int((a0["처리 일치"] & a0["사유 일치"]).sum()), int(a0["사람 확인"].sum())),
    ("범위밖 (D57 후)", len(a), int((a["처리 일치"] & a["사유 일치"]).sum()), int(a["사람 확인"].sum())),
    ("원천", len(b), int((b["원천 확인 필요"] == (b["사람"] == "오류")).sum()), 0),
    ("이관 (분류·판정 모두)", len(c7s), int((c7s["분류 일치"] & c7s["판정 일치"]).sum()), int((c7s["이관 판정"] == "사람 확인").sum())),
    ("이관 (결함 판정만)", len(c7s), int(c7s["판정 일치"].sum()), int((c7s["이관 판정"] == "사람 확인").sum())),
]
summ = pd.DataFrame(rows, columns=["표본", "크기", "일치", "사람 확인"])
summ["일치율(%)"] = (summ["일치"] / summ["크기"] * 100).round(1)
print(summ.to_string(index=False))
print()
print("범위밖 처리만 일치 — D57 전:", a0["처리 일치"].sum(), "/ 후:", a["처리 일치"].sum())
print("이관 혼동표 (분류):"); print(pd.crosstab(c7s["내 분류"], c7s["이관 분류"], margins=True))
print("이관 혼동표 (결함):"); print(pd.crosstab(c7s["내 판정"], c7s["이관 판정"], margins=True))

## 04 정리 (직접 작성)

1. 자동 규칙이 사람 판정과 다른 건은 어떤 유형인가? 규칙을 고칠지, 사람 판정을 계속 우선할지?

처음에는 자동 규칙이 범위 밖 32건 중 24건만 사람과 같은 처리를 냈다. 다른 건은 대부분 공개 기준으로는 범위 안이지만 AI가 스스로 적은 허용 범위를 벗어난 경우였다. 사람은 이걸 "자기가 정한 기준을 어긴 추론 결함"으로 보고 제외했는데, 규칙에는 이 조건이 없었다. 규칙을 추가하자(D57) 처리는 32건 모두 일치했다.
남은 차이는 사유 코드 6건이다.
- 고객 요청이나 경험을 이유로 범위 밖을 정당화한 사례(코드 1: 6429, 6866, 6878)와 원천 오류·부적합 정당화가 겹친 사례(6799)는 cot 문장의 맥락을 읽어야 판단할 수 있다.
- 6660은 사람은 코드 2, 규칙은 코드 4로 봤다. 문장 맥락이 아니라 성향과 허용 범위가 짝이 맞는지를 어떻게 읽느냐의 차이다.
- 194는 허용 범위 문장이 깨진 것을 규칙(R3-D57-DMG)이 찾아 사람에게 넘겼다. 원래 범위가 무엇이었는지 해석만 사람이 했다.

그래서 처리는 규칙으로 하고, 사유는 사람이 검토한 사례에 한해 사람 판정을 우선한다. 사람이 검토하지 않은 사례는 코드 1이 코드 2로 자동 분류되므로, 05의 사유 코드 분포에서 코드 1은 실제보다 적게 나온다.
이 32건은 규칙을 만들 때 참고한 표본이라 100% 일치는 일반화 성능이 아니라 "사람의 판정 기준을 규칙으로 옮겼다"는 뜻으로 본다.

2. body_grade와 원천검수표(H1 키워드 표본)가 서로 다른 오류를 잡는다면, 원천 확인 대상은 어떻게 정할 것인가?

원천 표본 20건은 키워드 불일치(H1) 후보에서 뽑은 것이다. 이 표본 안에서는 등급 비교(body_grade)가 오류 4건을 모두 잡고 오탐이 없었다. 반면 키워드 방식은 20건 중 4건만 실제 오류였다(16건 오탐).
다만 근거는 좁다. body_grade가 표시한 상품 319개 중 이 표본에 들어 있는 건 4개뿐이라 전체 오탐률은 측정하지 않았고, H1이 놓친 오류도 확인하지 못했다.
그래도 원천 확인 대상은 등록 등급과 설명 속 등급이 다른 상품을 기준으로 정한다. 설명에 등급 문장이 아예 없는 상품이 증권 상품 8,000개 중 3,722개(46.5%)라 등급 비교로는 잡을 수 없으므로, 이런 상품은 키워드 불일치를 보조 신호로 쓰고 추천 횟수가 많은 상품부터 확인한다.
최종 확인은 공식 자료와 대조해서 한다. 교보 공모주하이일드플러스는 body_grade가 표시했지만, 공식 자료로 보니 설명이 틀리고 등록 등급이 맞았다. 등급 비교는 어느 쪽이 맞는지까지 알려 주지 않는다. 이 대조를 B안에서 근거를 자동으로 붙이는 방식으로 줄인다.

3. 이관 규칙의 '사람 확인'과 불일치 사례 중 C안(LLM 1차 판정)으로 넘길 범위는?

규칙으로 충분한 것은 범위 검사, 원천 등급 비교, 등급과 요청 표현이 분명한 문장이다. C안으로 넘길 것은 문장의 뜻을 읽어야 하는 경우다.
- 사람 확인 103건: 특히 고객 성향 자체를 알 수 없는 경우(72건)와, 경험과 요청이 한 문장에 섞였는데 등급이 이름으로만 나온 경우
- 한 문장에 회피와 희망이 함께 있는 경우: cot_id 4224(한 문장에 높은 위험 회피와 4등급 상품 선택 희망이 함께 있음)에서 회피 표현이 문장 전체에 적용돼 4등급 요청까지 회피로 읽혔다. 문장 단위로 분류하는 규칙의 한계다.
- 고객 요청이나 경험을 이유로 범위 밖을 정당화한 추론(코드 1)

새 표본 20건의 결함 판정은 경험 규칙을 넣기 전 13건, 넣은 뒤 18건이 일치했다. 90%는 같은 표본을 보고 규칙을 고친 뒤의 수치라 낙관적이고, 20건이라 불확실성도 크다(대략 70~99%). 표본도 성향 밖 15건과 성향 안 5건으로 뽑아서 전체를 대표하지 않는다. 새 규칙의 실제 성능은 새 표본으로 다시 확인해야 한다. 4224 같은 사례에 맞춰 규칙을 더 고치지 않고, 한계로 기록해 C안 대상으로 둔다.